In [ ]:
#과제 3
import pandas as pd

# 교재 스타일의 Heap 클래스 구현
class Heap:
    def __init__(self):
        self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self.__percolateUp(len(self.__A) - 1)

    def __percolateUp(self, i):
        parent = (i - 1) // 2
        if i > 0 and self.__A[i] > self.__A[parent]:
            self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
            self.__percolateUp(parent)

    def deleteMax(self):
        if self.isEmpty(): return None
        max_val = self.__A[0]
        self.__A[0] = self.__A[len(self.__A) - 1]
        self.__A.pop()
        self.__percolateDown(0)
        return max_val

    def __percolateDown(self, i):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def isEmpty(self):
        return len(self.__A) == 0

# 데이터 로드 및 실행
df = pd.read_csv('birthday.csv')
df.columns = ['Name', 'Year', 'Month', 'Day']

h = Heap()
for _, row in df.iterrows():
    if pd.notna(row['Year']):
        # (년, 월, 일, 이름) 튜플로 저장하여 날짜 순으로 비교
        h.insert((int(row['Year']), int(row['Month']), int(row['Day']), row['Name']))

print("1. 생일이 느린 순서 10명:")
for i in range(1, 11):
    if h.isEmpty(): break
    v = h.deleteMax()
    print(f"[{i}] {v[3]}: {v[0]}-{v[1]:02d}-{v[2]:02d}")

1. 생일이 느린 순서 10명:
[1] 이윤서: 2006-12-27
[2] 정희원: 2006-12-21
[3] 김효린: 2006-12-16
[4] 이예은: 2006-12-09
[5] 김주영: 2006-11-20
[6] 전은빈: 2006-11-07
[7] 이하연: 2006-09-22
[8] 김우현: 2006-09-01
[9] 최윤지: 2006-08-30
[10] 유가현: 2006-08-22


In [ ]:
#과제 4
import pandas as pd

# --- 1. Heap 구현 (생일이 느린 순서 추출용) ---
class Heap:
    def __init__(self):
        self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self.__percolateUp(len(self.__A) - 1)

    def __percolateUp(self, i):
        parent = (i - 1) // 2
        if i > 0 and self.__A[i] > self.__A[parent]:
            self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
            self.__percolateUp(parent)

    def deleteMax(self):
        if self.isEmpty(): return None
        max_val = self.__A[0]
        self.__A[0] = self.__A[len(self.__A) - 1]
        self.__A.pop()
        self.__percolateDown(0)
        return max_val

    def __percolateDown(self, i):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def isEmpty(self):
        return len(self.__A) == 0

# --- 2. Circular Doubly Linked List 구현 (조원 관리용) ---
class BidirectNode:
    def __init__(self, x, prevNode, nextNode):
        self.item = x
        self.prev = prevNode
        self.next = nextNode

class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem):
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def getNode(self, i):
        curr = self.__head
        for _ in range(i + 1): curr = curr.next
        return curr

    def __iter__(self):
        curr = self.__head.next
        while curr != self.__head:
            yield curr.item
            curr = curr.next

# --- 데이터 처리 및 실행 ---

# 데이터 로드
df = pd.read_csv('birthday.csv')
df.columns = ['Name', 'Year', 'Month', 'Day']

# [Task 1 실행] 힙을 이용한 출력
print("### 1. 생일이 느린 순서 10명 (Heap 활용) ###")
h = Heap()
for _, row in df.iterrows():
    if pd.notna(row['Year']):
        h.insert((int(row['Year']), int(row['Month']), int(row['Day']), row['Name']))

for i in range(1, 11):
    v = h.deleteMax()
    if v: print(f"{i}. {v[3]} ({v[0]}-{v[1]:02d}-{v[2]:02d})")

# [Task 2 실행] 연결 리스트 및 개선된 예외 처리
print("\n### 2. 같은 조 친구들 명단 (연결 리스트 & 예외 처리) ###")
cdll = CircularDoublyLinkedList()
for _, row in df.iterrows():
    cdll.append(row.to_dict())

# 조원 명단 설정 (유지민 학생은 데이터가 비어있음)
my_group = ['안서영', '위주은', '강예은', '유지민', '김예준']

for person in cdll:
    name = person['Name']
    try:
        # 1. 조원인지 확인
        if name in my_group:
            # 2. 데이터가 유효한지 검사 (비어있으면 예외 발생)
            if pd.isna(person['Year']) or pd.isna(person['Month']) or pd.isna(person['Day']):
                raise ValueError("생년월일 정보 누락")
            
            # 3. 정상 데이터 출력
            birth_str = f"{int(person['Year'])}-{int(person['Month']):02d}-{int(person['Day']):02d}"
            print(f"[조원 확인] {name} | 생년월일: {birth_str}")
            
    except ValueError as e:
        # 조원이면서 데이터가 없는 경우에만 이 블록이 실행됨
        print(f"[데이터 오류] {name} | 사유: {e}")

### 1. 생일이 느린 순서 10명 (Heap 활용) ###
1. 이윤서 (2006-12-27)
2. 정희원 (2006-12-21)
3. 김효린 (2006-12-16)
4. 이예은 (2006-12-09)
5. 김주영 (2006-11-20)
6. 전은빈 (2006-11-07)
7. 이하연 (2006-09-22)
8. 김우현 (2006-09-01)
9. 최윤지 (2006-08-30)
10. 유가현 (2006-08-22)

### 2. 같은 조 친구들 명단 (연결 리스트 & 예외 처리) ###
[조원 확인] 안서영 | 생년월일: 2004-06-21
[조원 확인] 위주은 | 생년월일: 2001-01-12
[조원 확인] 강예은 | 생년월일: 2004-08-23
[조원 확인] 김예준 | 생년월일: 2003-03-30
[데이터 오류] 유지민 | 사유: 생년월일 정보 누락
